# Advanced Python Exceptions — Volume 2
## A guided, tutorial-style problem book with fully worked solutions

**Source and teaching approach.** Your supplied *Python Exceptions* lesson introduces exception objects, the `BaseException → Exception` hierarchy, matching subclasses in `except`, propagation and stack traces, and the danger of broad handlers. This **new** workbook uses those principles as its starting point, but develops new scenarios instead of recycling the previous workbook’s 32 solutions.

**What makes this a tutorial:** Every problem proceeds through a task, mental model, small experiment, error contract, incremental solution, detailed explanation, tests, and edge cases. Markdown discussion intentionally outnumbers code cells. Read the prediction before executing each experiment.

**Runtime:** Python 3.11 or newer (some problems use `ExceptionGroup`, `except*`, and `BaseException.add_note`). Standard library only. Execute cells top-to-bottom using **Run All**. Every intentional error is caught in a test/example, so the notebook finishes without uncaught exceptions.

## How to work through these exercises

1. Read the problem statement and write down the exception type you expect.
2. Pause at each **minimal experiment** and mentally predict its output.
3. Hide the **reference solution** cell if practicing independently; write your own implementation first.
4. Run the test cell. Add a failure case of your own before moving on.
5. Change one design decision and explain which error contract it changes.

**A key distinction:** The original lesson warns that `except Exception:` may hide programming bugs. Here, you will see exactly when a narrow handler is useful and when it is better not to catch an error at all. We use specific exception types and keep `try` blocks short. We do not treat traceback text as a stable API, because its formatting can differ between Python versions.

In [1]:
import sys
import traceback
from contextlib import contextmanager

assert sys.version_info >= (3, 11), 'Use Python 3.11+ for this tutorial.'
print('Python:', sys.version.split()[0])
print('Ready: all demonstration failures will be handled intentionally.')

Python: 3.13.7
Ready: all demonstration failures will be handled intentionally.


## Unit I — Exception objects, hierarchy, and precise control flow

These exercises extend the source lesson’s first concepts: an exception is an **object**; `raise` starts exception handling; an `except` clause matches an exception’s class **and its subclasses**. We begin with experiments that reveal the behavior before designing APIs.

### Problem 01 — Build a diagnostic exception object without raising it

**The task.** Create a function that prepares a `ValueError` with a message and structured arguments but does **not** throw it. Inspect object identity before and after raising it. Then verify the exception's arguments remain intact.

**Your goal before reading the solution:** decide which operation can fail, which exception communicates that failure, and which layer should handle it. The next cells work through the solution in small pieces.

**Step 1 — Construct the mental model.**

Instantiation and raising are separate operations. `ValueError('bad', 7)` is an object whose `.args` is `('bad', 7)`. Only `raise error` initiates stack unwinding. Holding an exception object for too long **after** it has been raised can retain traceback frames; clear stored tracebacks if you must cache it.

**Step 2 — Run a minimal experiment.**

Predict what happens before executing the cell. Unlike deliberately crashing a notebook, we capture expected exceptions so **Run All** remains usable.

In [2]:
prepared = ValueError('out-of-range', 7)
print('constructed:', type(prepared).__name__, prepared.args)
print('traceback before raising:', prepared.__traceback__ is None)
try:
    raise prepared
except ValueError as caught:
    print('same object:', caught is prepared)
    print('traceback after raising:', caught.__traceback__ is not None)

constructed: ValueError ('out-of-range', 7)
traceback before raising: True
same object: True
traceback after raising: True


**Step 3 — Design the error contract.**

Return the object as data, not a raised condition. When subsequently raising it, the same object travels through the handler. Build exceptions close to their throw sites in production so they do not unnecessarily retain traceback references.

**Step 4 — Build the reference solution incrementally.**

Read the function signature first, then trace the normal path, then locate each `raise` and `except`. Comments mark the decision boundaries.

In [3]:
def prepare_invalid_age(value: object) -> ValueError:
    # Nothing is raised here: callers explicitly choose when to raise.
    return ValueError('age must be between 0 and 130', value)

def validate_age(value: int) -> int:
    if type(value) is not int or not 0 <= value <= 130:
        raise prepare_invalid_age(value)
    return value

**Step 5 — Walk through why the solution works.**

The factory creates an ordinary exception instance. `validate_age` is the boundary that raises it. The strict `type(value) is int` is an explicit contract here: booleans are subclasses of `int` but do not count as ages. Notice that the error's two arguments are kept separately, while `str(error)` is only a display representation.

**Step 6 — Exercise normal, boundary, and failure paths.**

These assertions are executable specifications, not just illustrations. If you edit the solution, rerun this cell.

In [4]:
assert validate_age(0) == 0
assert validate_age(130) == 130
for invalid in (-1, 131, True, '20'):
    try:
        validate_age(invalid)
    except ValueError as error:
        assert error.args == ('age must be between 0 and 130', invalid)
    else:
        raise AssertionError(f'expected ValueError for {invalid!r}')
print('Problem 1: valid values and four error cases passed.')

Problem 1: valid values and four error cases passed.


**Step 7 — Edge cases and engineering trade-offs.**

A stored, previously raised exception may keep `.__traceback__` references. Prefer storing lightweight error records rather than living exception objects in long-lived caches. The two-argument message is deliberate so callers can inspect `.args`; a user-facing message can be formatted independently.

**Step 8 — One more experiment / stretch solution.**

Compare this variant with the first solution, then run its checks.

In [5]:
instance = prepare_invalid_age(999)
assert isinstance(instance, ValueError)
assert isinstance(instance, Exception)
assert isinstance(instance, BaseException)
assert instance.__traceback__ is None
print('A constructed instance does not automatically raise.')

A constructed instance does not automatically raise.


**Takeaway.** Creating an exception is not equivalent to raising it; document the exact boundary that makes failure observable.

---

### Problem 02 — Prove which except branch runs first

**The task.** Write a classifier for `IndexError`, `KeyError`, and other `LookupError` subclasses, then demonstrate how a badly ordered handler masks a more specific one.

**Your goal before reading the solution:** decide which operation can fail, which exception communicates that failure, and which layer should handle it. The next cells work through the solution in small pieces.

**Step 1 — Construct the mental model.**

`IndexError` and `KeyError` inherit from `LookupError`. An `except LookupError` above `except IndexError` catches an `IndexError` first, making the later branch unreachable for that error. Python does not search for the most specific handler: it chooses the **first matching** one in source order.

**Step 2 — Run a minimal experiment.**

Predict what happens before executing the cell. Unlike deliberately crashing a notebook, we capture expected exceptions so **Run All** remains usable.

In [6]:
def handler_order_probe():
    try:
        raise IndexError('missing position')
    except LookupError:
        return 'general handler ran'
    except IndexError:
        return 'unreachable specific handler'

print(handler_order_probe())
assert handler_order_probe() == 'general handler ran'

general handler ran


**Step 3 — Design the error contract.**

Place subclasses ahead of their base class. Do not route all exceptions through `Exception` just to print their names. Return distinct domain-relevant outcomes from each handler.

**Step 4 — Build the reference solution incrementally.**

Read the function signature first, then trace the normal path, then locate each `raise` and `except`. Comments mark the decision boundaries.

In [7]:
def classify_lookup(operation):
    try:
        operation()
    except IndexError:
        return 'invalid index'
    except KeyError:
        return 'missing key'
    except LookupError:
        return 'other lookup failure'
    else:
        return 'success'

**Step 5 — Walk through why the solution works.**

For an `IndexError`, the first branch matches. For a `KeyError`, the first fails and the second matches. Another custom `LookupError` reaches the third. A `TypeError` is not converted into a lookup failure; it is allowed to propagate. The `else` expresses the success path distinctly.

**Step 6 — Exercise normal, boundary, and failure paths.**

These assertions are executable specifications, not just illustrations. If you edit the solution, rerun this cell.

In [8]:
def fails(exc):
    def operation():
        raise exc
    return operation

assert classify_lookup(lambda: 42) == 'success'
assert classify_lookup(fails(IndexError())) == 'invalid index'
assert classify_lookup(fails(KeyError('x'))) == 'missing key'
assert classify_lookup(fails(LookupError('other'))) == 'other lookup failure'
try:
    classify_lookup(fails(TypeError('bug')))
except TypeError as error:
    assert str(error) == 'bug'
else:
    raise AssertionError('TypeError must propagate')
print('Problem 2: all dispatch paths passed.')

Problem 2: all dispatch paths passed.


**Step 7 — Edge cases and engineering trade-offs.**

Multiple `except` clauses are checked in order, not by inheritance depth. Never use `except BaseException` for ordinary application errors: it also covers control-flow exceptions such as `KeyboardInterrupt` and `SystemExit`.

**Step 8 — One more experiment / stretch solution.**

Compare this variant with the first solution, then run its checks.

In [9]:
class CustomLookupFailure(LookupError):
    pass

assert classify_lookup(fails(CustomLookupFailure())) == 'other lookup failure'
print('A newly introduced subclass still uses the general lookup handler.')

A newly introduced subclass still uses the general lookup handler.


**Takeaway.** Handle only the failure you can meaningfully resolve, and let unrelated exceptions remain visible.

---

### Problem 03 — Keep a try block narrow enough to expose a formatting bug

**The task.** Implement a ratio-reporting function. A zero denominator should produce an explicit fallback; a broken formatter should remain an error. Contrast a broad `try` block with a narrow one.

**Your goal before reading the solution:** decide which operation can fail, which exception communicates that failure, and which layer should handle it. The next cells work through the solution in small pieces.

**Step 1 — Construct the mental model.**

A handler can catch an exception thrown by **any statement inside its corresponding `try` suite**. If unrelated formatting runs inside a `try` for division, a handler may mistakenly blame formatting failure on arithmetic. Prefer a short `try` around the operation you intend to recover from.

**Step 2 — Run a minimal experiment.**

Predict what happens before executing the cell. Unlike deliberately crashing a notebook, we capture expected exceptions so **Run All** remains usable.

In [10]:
def broken_report(value):
    return value + ' units'   # floats and strings cannot be added

try:
    number = 8 / 2
    broken_report(number)
except TypeError as error:
    print('Formatting raised:', type(error).__name__)

Formatting raised: TypeError


**Step 3 — Design the error contract.**

The error contract is: zero divisor → fallback string; ordinary valid division → formatter output; formatter defects → `TypeError` propagates. Put formatting after the `try`/`except`/`else` statement rather than trying to recover from it.

**Step 4 — Build the reference solution incrementally.**

Read the function signature first, then trace the normal path, then locate each `raise` and `except`. Comments mark the decision boundaries.

In [11]:
def ratio_report(numerator, denominator, formatter=str):
    try:
        ratio = numerator / denominator
    except ZeroDivisionError:
        return 'ratio unavailable: zero denominator'
    # The formatter runs OUTSIDE the arithmetic error handler.
    return formatter(ratio)

**Step 5 — Walk through why the solution works.**

An explicit `ZeroDivisionError` branch is precise. Even a buggy formatter that raises `ZeroDivisionError` is outside the `try` suite and therefore propagates, which helps diagnose the true source. The formatter is injectable so the failure boundary can be tested without outside dependencies.

**Step 6 — Exercise normal, boundary, and failure paths.**

These assertions are executable specifications, not just illustrations. If you edit the solution, rerun this cell.

In [12]:
assert ratio_report(8, 2) == '4.0'
assert ratio_report(8, 0) == 'ratio unavailable: zero denominator'
assert ratio_report(8, 2, lambda n: f'{n:.1f}x') == '4.0x'
try:
    ratio_report(8, 2, broken_report)
except TypeError:
    pass
else:
    raise AssertionError('Formatting defect must not be suppressed')
print('Problem 3: recovery and propagation both confirmed.')

Problem 3: recovery and propagation both confirmed.


**Step 7 — Edge cases and engineering trade-offs.**

Do not replace every possible arithmetic error with a fallback. For example, `8 / '2'` raises a `TypeError`, which signals invalid input under this contract. A separate input-validation layer could handle that explicitly.

**Step 8 — One more experiment / stretch solution.**

Compare this variant with the first solution, then run its checks.

In [13]:
def formatter_with_zero_division(value):
    return 1 / 0

try:
    ratio_report(8, 2, formatter_with_zero_division)
except ZeroDivisionError:
    print('The error came from formatter and remains visible.')

The error came from formatter and remains visible.


**Takeaway.** Handle only the failure you can meaningfully resolve, and let unrelated exceptions remain visible.

---

### Problem 04 — Determine which exceptions try/except/else/finally can catch

**The task.** Instrument a four-clause `try` statement. Demonstrate that the `except` handles a failure in `try`, but does **not** handle an exception newly raised inside the sibling `else`. Verify `finally` still executes.

**Your goal before reading the solution:** decide which operation can fail, which exception communicates that failure, and which layer should handle it. The next cells work through the solution in small pieces.

**Step 1 — Construct the mental model.**

`else` executes only if the `try` suite finishes normally (not when it returns, breaks, continues, or raises). An exception raised inside that `else` does not jump backwards to an `except` belonging to the same statement. The `finally` suite runs as the statement leaves, barring abrupt process termination.

**Step 2 — Run a minimal experiment.**

Predict what happens before executing the cell. Unlike deliberately crashing a notebook, we capture expected exceptions so **Run All** remains usable.

In [14]:
events = []
try:
    try:
        events.append('try')
    except ValueError:
        events.append('except')
    else:
        events.append('else')
        raise ValueError('raised in else')
    finally:
        events.append('finally')
except ValueError:
    events.append('outer handler')
print(events)

['try', 'else', 'finally', 'outer handler']


**Step 3 — Design the error contract.**

Use a helper returning an event log. It deliberately nests an outer handler so the notebook does not fail. Parameterize where the failure originates to distinguish the branches.

**Step 4 — Build the reference solution incrementally.**

Read the function signature first, then trace the normal path, then locate each `raise` and `except`. Comments mark the decision boundaries.

In [15]:
def clause_trace(fail_in=None):
    log = []
    try:
        try:
            log.append('try')
            if fail_in == 'try':
                raise ValueError('from try')
        except ValueError:
            log.append('except')
        else:
            log.append('else')
            if fail_in == 'else':
                raise ValueError('from else')
        finally:
            log.append('finally')
    except ValueError:
        log.append('outer except')
    return log

**Step 5 — Walk through why the solution works.**

The inner handler catches only a `ValueError` from the inner `try`. A `ValueError` from `else` propagates through `finally` to the outer handler. In a clean run, the sequence is `try`, `else`, `finally`. This is a direct control-flow model, not a claim that every `finally` is guaranteed under power loss or process termination.

**Step 6 — Exercise normal, boundary, and failure paths.**

These assertions are executable specifications, not just illustrations. If you edit the solution, rerun this cell.

In [16]:
assert clause_trace() == ['try', 'else', 'finally']
assert clause_trace('try') == ['try', 'except', 'finally']
assert clause_trace('else') == ['try', 'else', 'finally', 'outer except']
for target in (None, 'try', 'else'):
    print(f'{target!r:>6}: {clause_trace(target)}')

  None: ['try', 'else', 'finally']
 'try': ['try', 'except', 'finally']
'else': ['try', 'else', 'finally', 'outer except']


**Step 7 — Edge cases and engineering trade-offs.**

Put only the risky operation inside the `try`, and place success-only operations in `else` when it clarifies intent. A failure in `finally` can replace an in-flight exception; problem 6 investigates that separately.

**Takeaway.** Handle only the failure you can meaningfully resolve, and let unrelated exceptions remain visible.

---

### Problem 05 — Trace a failure through four layers without swallowing it

**The task.** Build a miniature call chain: `source → transform → service → interface`. The innermost layer raises `ValueError`; the middle layers do not catch it; the interface inspects the traceback and returns a diagnostic dictionary.

**Your goal before reading the solution:** decide which operation can fail, which exception communicates that failure, and which layer should handle it. The next cells work through the solution in small pieces.

**Step 1 — Construct the mental model.**

An exception propagates up the active call stack until a matching handler is found. A traceback records the frames through which the exception traveled. Frame line numbers can vary by runtime, so compare function names instead of exact formatted text.

**Step 2 — Run a minimal experiment.**

Predict what happens before executing the cell. Unlike deliberately crashing a notebook, we capture expected exceptions so **Run All** remains usable.

In [17]:
def probe_inner():
    raise ValueError('invalid payload')
def probe_middle():
    return probe_inner()
try:
    probe_middle()
except ValueError as error:
    names = [frame.name for frame in traceback.extract_tb(error.__traceback__)]
    print('frame names:', names)
    assert names[-2:] == ['probe_middle', 'probe_inner']

frame names: ['<module>', 'probe_middle', 'probe_inner']


**Step 3 — Design the error contract.**

Keep the inner functions unaware of user-interface concerns. Only the outermost layer can format the diagnostic. Extract stable information using the `traceback` module, not brittle string parsing of the full traceback.

**Step 4 — Build the reference solution incrementally.**

Read the function signature first, then trace the normal path, then locate each `raise` and `except`. Comments mark the decision boundaries.

In [18]:
def tutorial_source():
    raise ValueError('malformed input')

def tutorial_transform():
    return tutorial_source()

def tutorial_service():
    return tutorial_transform()

def tutorial_interface():
    try:
        tutorial_service()
    except ValueError as error:
        frames = traceback.extract_tb(error.__traceback__)
        return {'exception': type(error).__name__,
                'message': str(error),
                'call_chain': [frame.name for frame in frames]}
    raise AssertionError('This example intentionally expects a failure')

**Step 5 — Walk through why the solution works.**

The `source` raises. `transform` and `service` simply propagate the exception; the interface handles it once. A frame list includes the handler’s invocation frame followed by each nested function. Diagnosing at the boundary preserves context and avoids duplicating log messages at every layer.

**Step 6 — Exercise normal, boundary, and failure paths.**

These assertions are executable specifications, not just illustrations. If you edit the solution, rerun this cell.

In [19]:
diagnostic = tutorial_interface()
assert diagnostic['exception'] == 'ValueError'
assert diagnostic['message'] == 'malformed input'
assert diagnostic['call_chain'][-4:] == [
    'tutorial_interface', 'tutorial_service',
    'tutorial_transform', 'tutorial_source']
print('Captured chain:', ' → '.join(diagnostic['call_chain'][-4:]))

Captured chain: tutorial_interface → tutorial_service → tutorial_transform → tutorial_source


**Step 7 — Edge cases and engineering trade-offs.**

Do not expose raw tracebacks to untrusted end users: they may contain file paths, function names, or sensitive data. A diagnostic dictionary is intended for a trusted developer-facing surface in this exercise. Tracebacks attached to retained exception objects can keep local variables alive.

**Takeaway.** Handle only the failure you can meaningfully resolve, and let unrelated exceptions remain visible.

---

### Problem 06 — Repair a finally clause that hides the real failure

**The task.** Demonstrate how `return` in `finally` can suppress a pending exception. Refactor a function that must record cleanup while preserving its success result or original failure.

**Your goal before reading the solution:** decide which operation can fail, which exception communicates that failure, and which layer should handle it. The next cells work through the solution in small pieces.

**Step 1 — Construct the mental model.**

A `finally` clause runs even when the `try` is returning or raising. Returning from `finally` overrides both the pending return value and the pending exception. This surprising behavior is one reason `finally` should usually be limited to cleanup, without `return`, `break`, or `continue`.

**Step 2 — Run a minimal experiment.**

Predict what happens before executing the cell. Unlike deliberately crashing a notebook, we capture expected exceptions so **Run All** remains usable.

In [20]:
def problematic_finally():
    try:
        raise ValueError('important failure')
    finally:
        return 'masked!'

assert problematic_finally() == 'masked!'
print('A return from finally concealed the ValueError.')

A return from finally concealed the ValueError.


**Step 3 — Design the error contract.**

Track cleanup events with a caller-supplied list. Return only from the normal path and allow the exception path to propagate automatically. `finally` appends the cleanup marker, and nothing else.

**Step 4 — Build the reference solution incrementally.**

Read the function signature first, then trace the normal path, then locate each `raise` and `except`. Comments mark the decision boundaries.

In [21]:
def perform_with_cleanup(should_fail: bool, events: list[str]):
    try:
        events.append('work')
        if should_fail:
            raise ValueError('work failed')
        return 'result'
    finally:
        events.append('cleanup')

**Step 5 — Walk through why the solution works.**

The normal return is preserved because `finally` merely appends. On failure, Python enters `finally`, appends the marker, and resumes propagating the `ValueError`. This pattern gives one cleanup site while keeping the original exception observable.

**Step 6 — Exercise normal, boundary, and failure paths.**

These assertions are executable specifications, not just illustrations. If you edit the solution, rerun this cell.

In [22]:
events = []
assert perform_with_cleanup(False, events) == 'result'
assert events == ['work', 'cleanup']
events = []
try:
    perform_with_cleanup(True, events)
except ValueError as error:
    assert str(error) == 'work failed'
else:
    raise AssertionError('ValueError was lost')
assert events == ['work', 'cleanup']
print('Problem 6: cleanup runs; return and exception both preserved.')

Problem 6: cleanup runs; return and exception both preserved.


**Step 7 — Edge cases and engineering trade-offs.**

Cleanup code itself can fail, in which case exception handling becomes more complicated. For reliable resource management prefer context managers and decide explicitly whether/how to surface a cleanup failure. Do not silently replace the original error with an unconditional `return`.

**Takeaway.** Handle only the failure you can meaningfully resolve, and let unrelated exceptions remain visible.

---

### Problem 07 — Compare bare raise with raise error using traceback frames

**The task.** Write two forwarding functions. One uses a bare `raise` in an active handler; the other uses `raise error`. Inspect the traceback frames and explain which choice best expresses re-propagation.

**Your goal before reading the solution:** decide which operation can fail, which exception communicates that failure, and which layer should handle it. The next cells work through the solution in small pieces.

**Step 1 — Construct the mental model.**

`raise` without an argument re-raises the active exception with its existing traceback. `raise error` raises the exception instance again from the current line, generally adding another traceback entry. Both preserve the exception type and instance, but the bare form communicates intent and avoids an extra re-raise site.

**Step 2 — Run a minimal experiment.**

Predict what happens before executing the cell. Unlike deliberately crashing a notebook, we capture expected exceptions so **Run All** remains usable.

In [23]:
def root_for_reraise():
    raise ValueError('root')
def forward_bare():
    try:
        root_for_reraise()
    except ValueError:
        raise

def forward_named():
    try:
        root_for_reraise()
    except ValueError as error:
        raise error

for function in (forward_bare, forward_named):
    try:
        function()
    except ValueError as error:
        print(function.__name__,
              [f.name for f in traceback.extract_tb(error.__traceback__)])

forward_bare ['<module>', 'forward_bare', 'root_for_reraise']
forward_named ['<module>', 'forward_named', 'forward_named', 'root_for_reraise']


**Step 3 — Design the error contract.**

Make a small inspection helper. Examine the `__traceback__` object after the exception is caught at an external boundary; do not depend on IPython’s rendered exception display.

**Step 4 — Build the reference solution incrementally.**

Read the function signature first, then trace the normal path, then locate each `raise` and `except`. Comments mark the decision boundaries.

In [24]:
def traceback_names(callable_):
    try:
        callable_()
    except ValueError as error:
        return [f.name for f in traceback.extract_tb(error.__traceback__)]
    raise AssertionError('Expected a ValueError')

**Step 5 — Walk through why the solution works.**

With the bare re-raise, the handler’s existing frame is not added a second time. With `raise error`, the handler’s line becomes an additional traceback frame. The actual lines and their rendering are implementation/version details, but the causal origin is still present in both cases.

**Step 6 — Exercise normal, boundary, and failure paths.**

These assertions are executable specifications, not just illustrations. If you edit the solution, rerun this cell.

In [25]:
bare_names = traceback_names(forward_bare)
named_names = traceback_names(forward_named)
assert bare_names[-1] == 'root_for_reraise'
assert named_names[-1] == 'root_for_reraise'
assert named_names.count('forward_named') >= bare_names.count('forward_bare')
assert len(named_names) >= len(bare_names)
print('Bare raise:', bare_names)
print('Named raise:', named_names)

Bare raise: ['traceback_names', 'forward_bare', 'root_for_reraise']
Named raise: ['traceback_names', 'forward_named', 'forward_named', 'root_for_reraise']


**Step 7 — Edge cases and engineering trade-offs.**

A bare `raise` is valid only when an exception is currently being handled; outside a handler it raises `RuntimeError`. Use `raise NewError(...) from error` instead when you intentionally translate the error into another API-level type.

**Takeaway.** Handle only the failure you can meaningfully resolve, and let unrelated exceptions remain visible.

---

## Unit II — Translation, validation, and iterator boundaries

The source lecture shows how catching `Exception` can hide `TypeError` bugs in a generator. The next problems apply that principle to parser boundaries, row processing, iterator protocols, and configuration lookups. Each exercise distinguishes *expected bad input* from *a bug in the implementation*.

### Problem 08 — Translate a low-level parse failure with line and column information

**The task.** Parse a semicolon-delimited list of integers like `10; -3; 5`. On invalid tokens, raise a custom `IntegerFieldError` exposing a one-based field index and chaining the original `ValueError`. Reject empty tokens, including trailing delimiters.

**Your goal before reading the solution:** decide which operation can fail, which exception communicates that failure, and which layer should handle it. The next cells work through the solution in small pieces.

**Step 1 — Construct the mental model.**

`int(token)` can raise `ValueError` for bad text. A parsing API can translate that into a domain error that identifies the field without destroying the underlying cause. `raise DomainError(...) from original` sets `__cause__` to the original exception and makes the translation explicit.

**Step 2 — Run a minimal experiment.**

Predict what happens before executing the cell. Unlike deliberately crashing a notebook, we capture expected exceptions so **Run All** remains usable.

In [26]:
try:
    int('three')
except ValueError as original:
    translated = RuntimeError('field 2 is not an integer')
    translated.__cause__ = original  # inspect relationship without raising
    print(type(translated.__cause__).__name__, str(original))

ValueError invalid literal for int() with base 10: 'three'


**Step 3 — Design the error contract.**

Introduce one custom subclass of `ValueError`, because the problem is invalid input value. Parse each trimmed token. Limit the `try` suite to `int(token)` and construct the domain error in its `except` block. Keep the index and raw token as attributes.

**Step 4 — Build the reference solution incrementally.**

Read the function signature first, then trace the normal path, then locate each `raise` and `except`. Comments mark the decision boundaries.

In [27]:
class IntegerFieldError(ValueError):
    def __init__(self, field_index: int, token: str):
        self.field_index = field_index
        self.token = token
        super().__init__(f'field {field_index}: invalid integer {token!r}')

def parse_integer_fields(text: str) -> list[int]:
    if not isinstance(text, str):
        raise TypeError('text must be str')
    parsed = []
    for index, raw in enumerate(text.split(';'), start=1):
        token = raw.strip()
        try:
            value = int(token)
        except ValueError as original:
            raise IntegerFieldError(index, raw) from original
        parsed.append(value)
    return parsed

**Step 5 — Walk through why the solution works.**

The function starts by rejecting the wrong argument type, rather than treating it as malformed user text. Splitting an empty string yields one empty token, which is invalid. Chaining preserves the original conversion error for debugging. The public exception provides stable `field_index` and `token` attributes; callers should prefer them over parsing its message.

**Step 6 — Exercise normal, boundary, and failure paths.**

These assertions are executable specifications, not just illustrations. If you edit the solution, rerun this cell.

In [28]:
assert parse_integer_fields('10; -3; 5') == [10, -3, 5]
assert parse_integer_fields(' 0 ; +8 ') == [0, 8]
for sample, expected_index in [('1;no;3', 2), ('1;', 2), ('', 1)]:
    try:
        parse_integer_fields(sample)
    except IntegerFieldError as error:
        assert error.field_index == expected_index
        assert isinstance(error.__cause__, ValueError)
    else:
        raise AssertionError(f'should reject {sample!r}')
try:
    parse_integer_fields(None)
except TypeError:
    pass
else:
    raise AssertionError('wrong type must be visible')
print('Problem 8: parse contracts and chaining passed.')

Problem 8: parse contracts and chaining passed.


**Step 7 — Edge cases and engineering trade-offs.**

This example does not support quoted separators or thousands separators; those features would require a different grammar. The custom error is a subclass of `ValueError`, so callers may catch either the detailed domain class or its broader category.

**Step 8 — One more experiment / stretch solution.**

Compare this variant with the first solution, then run its checks.

In [29]:
try:
    parse_integer_fields('9;broken')
except IntegerFieldError as error:
    print('user-facing:', error)
    print('developer-facing cause:', repr(error.__cause__))

user-facing: field 2: invalid integer 'broken'
developer-facing cause: ValueError("invalid literal for int() with base 10: 'broken'")


**Takeaway.** Handle only the failure you can meaningfully resolve, and let unrelated exceptions remain visible.

---

### Problem 09 — Handle expected bad rows without swallowing programming defects

**The task.** Process `(name, quantity)` records. Accept positive integer quantities; report invalid values per row; abort with an exception if a record has the wrong structural shape. Return successes and errors separately.

**Your goal before reading the solution:** decide which operation can fail, which exception communicates that failure, and which layer should handle it. The next cells work through the solution in small pieces.

**Step 1 — Construct the mental model.**

A business rule violation such as quantity `-2` is expected bad data. Unpacking a record with the wrong number of values is a structural error under this contract and should not be hidden. Validate quantities explicitly and catch only a dedicated `InvalidQuantity` exception for recoverable row errors.

**Step 2 — Run a minimal experiment.**

Predict what happens before executing the cell. Unlike deliberately crashing a notebook, we capture expected exceptions so **Run All** remains usable.

In [30]:
rows = [('paper', 3), ('pens', -2), ('tape', 1)]
for name, quantity in rows:
    print(name, 'good' if type(quantity) is int and quantity > 0 else 'bad quantity')

paper good
pens bad quantity
tape good


**Step 3 — Design the error contract.**

Make validation a separate function that raises one recognizable exception. Put the narrow handler around validation only, not around record unpacking or success-list manipulation. Return a structured `(accepted, rejected)` pair.

**Step 4 — Build the reference solution incrementally.**

Read the function signature first, then trace the normal path, then locate each `raise` and `except`. Comments mark the decision boundaries.

In [31]:
class InvalidQuantity(ValueError):
    pass

def require_quantity(value):
    if type(value) is not int or value <= 0:
        raise InvalidQuantity(f'quantity must be a positive integer: {value!r}')
    return value

def inspect_inventory(rows):
    accepted, rejected = [], []
    for row_number, row in enumerate(rows, 1):
        name, quantity = row  # structural mistakes are not swallowed
        try:
            count = require_quantity(quantity)
        except InvalidQuantity as error:
            rejected.append((row_number, name, str(error)))
            continue
        accepted.append((name, count))
    return accepted, rejected

**Step 5 — Walk through why the solution works.**

The `try` encloses exactly one operation, `require_quantity`. A recoverable error generates a row-specific report; `continue` advances to the next row. A malformed row raises `ValueError` during unpacking outside the `try`. This makes structural regressions visible rather than fabricating a partial success.

**Step 6 — Exercise normal, boundary, and failure paths.**

These assertions are executable specifications, not just illustrations. If you edit the solution, rerun this cell.

In [32]:
good, bad = inspect_inventory([('paper', 3), ('pens', -2), ('tape', 1), ('eraser', True)])
assert good == [('paper', 3), ('tape', 1)]
assert [item[0] for item in bad] == [2, 4]
try:
    inspect_inventory([('fine', 1), ('missing_quantity',)])
except ValueError:
    pass
else:
    raise AssertionError('malformed row must propagate')
print('Problem 9:', len(good), 'accepted;', len(bad), 'rejected.')

Problem 9: 2 accepted; 2 rejected.


**Step 7 — Edge cases and engineering trade-offs.**

Do not catch a broad `ValueError` around both unpacking and quantity checking; it would merge different defects. If malformed shape is expected user input in a different application, define a separate `InvalidRecordShape` path intentionally.

**Takeaway.** Handle only the failure you can meaningfully resolve, and let unrelated exceptions remain visible.

---

### Problem 10 — Understand StopIteration inside a generator

**The task.** Write a generator that groups items into exact pairs. A trailing unpaired item must raise a domain-specific `ValueError`, not accidentally leak `StopIteration` from inside generator code.

**Your goal before reading the solution:** decide which operation can fail, which exception communicates that failure, and which layer should handle it. The next cells work through the solution in small pieces.

**Step 1 — Construct the mental model.**

At a generator boundary, `next(iterator)` raises `StopIteration` when exhausted. If an unhandled `StopIteration` escapes a generator body, modern Python transforms it into `RuntimeError` (PEP 479). Catch exhaustion close to the `next()` call and translate only when an item is missing mid-pair.

**Step 2 — Run a minimal experiment.**

Predict what happens before executing the cell. Unlike deliberately crashing a notebook, we capture expected exceptions so **Run All** remains usable.

In [33]:
def incorrectly_pair(items):
    iterator = iter(items)
    while True:
        first = next(iterator)  # unhandled exhaustion inside a generator
        second = next(iterator)
        yield first, second

try:
    list(incorrectly_pair([1, 2]))
except RuntimeError as error:
    print('unhandled StopIteration became:', type(error).__name__)
    assert isinstance(error.__cause__, StopIteration)

unhandled StopIteration became: RuntimeError


**Step 3 — Design the error contract.**

Instead of using `next()` at the start of each pair, iterate over first items using a `for` loop, which handles normal exhaustion correctly. A missing *second* item is an error: catch that one `StopIteration` and translate to `ValueError` with an explicit cause.

**Step 4 — Build the reference solution incrementally.**

Read the function signature first, then trace the normal path, then locate each `raise` and `except`. Comments mark the decision boundaries.

In [34]:
def exact_pairs(items):
    iterator = iter(items)
    for first in iterator:
        try:
            second = next(iterator)
        except StopIteration as original:
            raise ValueError('input has an unpaired final item') from original
        yield first, second

**Step 5 — Walk through why the solution works.**

The `for` loop consumes first elements and ends cleanly when none remain. Only an incomplete pair triggers the `except StopIteration` branch. The exception handler does not surround `yield`, so exceptions thrown by consumer code cannot accidentally be mistaken for iterator exhaustion.

**Step 6 — Exercise normal, boundary, and failure paths.**

These assertions are executable specifications, not just illustrations. If you edit the solution, rerun this cell.

In [35]:
assert list(exact_pairs([])) == []
assert list(exact_pairs([1, 2, 3, 4])) == [(1, 2), (3, 4)]
try:
    list(exact_pairs([1, 2, 3]))
except ValueError as error:
    assert 'unpaired' in str(error)
    assert isinstance(error.__cause__, StopIteration)
else:
    raise AssertionError('odd-length input must fail')
print('Problem 10: even and odd streams behave correctly.')

Problem 10: even and odd streams behave correctly.


**Step 7 — Edge cases and engineering trade-offs.**

A one-shot iterator is consumed as you iterate: callers should not assume it can be reused. If the policy instead requires keeping a leftover item, return a different interface that explicitly reports it. No broad `Exception` handler is necessary.

**Step 8 — One more experiment / stretch solution.**

Compare this variant with the first solution, then run its checks.

In [36]:
def squares_up_to_three():
    for x in range(1, 4):
        yield x*x
# Three generated squares are an odd-length input.
# The next check confirms that the trailing value is not silently discarded.
try:
    list(exact_pairs(squares_up_to_three()))
except ValueError:
    print('Generator input correctly reports the incomplete pair.')

Generator input correctly reports the incomplete pair.


**Takeaway.** Handle only the failure you can meaningfully resolve, and let unrelated exceptions remain visible.

---

### Problem 11 — Guarantee a collection is unchanged after failed validation

**The task.** Implement `append_unique_batch(existing, proposed)` that atomically appends names, rejecting duplicates in either the original list or proposed batch. On failure, the original list must remain unchanged.

**Your goal before reading the solution:** decide which operation can fail, which exception communicates that failure, and which layer should handle it. The next cells work through the solution in small pieces.

**Step 1 — Construct the mental model.**

A loop that mutates the destination and then detects a duplicate leaves partial work behind. This is an exception-safety problem, not just an exception-type problem. Validate against a separate `seen` set first, then perform one commit after all validation succeeds.

**Step 2 — Run a minimal experiment.**

Predict what happens before executing the cell. Unlike deliberately crashing a notebook, we capture expected exceptions so **Run All** remains usable.

In [37]:
naive = ['alice']
try:
    for name in ['bob', 'alice']:
        if name in naive:
            raise ValueError(f'duplicate {name}')
        naive.append(name)
except ValueError:
    print('naive mutated state:', naive)
assert naive == ['alice', 'bob']

naive mutated state: ['alice', 'bob']


**Step 3 — Design the error contract.**

Define a *strong exception guarantee*: the passed-in list is either fully updated or unchanged when a validation error occurs. Materialize the proposed input once, validate it without altering the original, and append in one slice operation.

**Step 4 — Build the reference solution incrementally.**

Read the function signature first, then trace the normal path, then locate each `raise` and `except`. Comments mark the decision boundaries.

In [38]:
def append_unique_batch(existing: list[str], proposed) -> list[str]:
    additions = list(proposed)
    seen = set(existing)
    if len(seen) != len(existing):
        raise ValueError('existing names already contain duplicates')
    for name in additions:
        if not isinstance(name, str):
            raise TypeError('all names must be strings')
        if name in seen:
            raise ValueError(f'duplicate name: {name!r}')
        seen.add(name)
    existing.extend(additions)  # single commit AFTER checks
    return existing

**Step 5 — Walk through why the solution works.**

The validation loop changes only its private `seen` set. A duplicate or wrong type raises before `extend` is reached, preserving the caller’s list. `extend` itself is one commit operation but is not a universal transactional guarantee against failures such as allocation errors or arbitrary subclasses of `list`; the defined guarantee is for validation failures with an ordinary list.

**Step 6 — Exercise normal, boundary, and failure paths.**

These assertions are executable specifications, not just illustrations. If you edit the solution, rerun this cell.

In [39]:
names = ['alice']
assert append_unique_batch(names, ['bob', 'carol']) is names
assert names == ['alice', 'bob', 'carol']
for incoming, error_type in [(['dave', 'alice'], ValueError), (['dave', 42], TypeError)]:
    original = names.copy()
    try:
        append_unique_batch(names, incoming)
    except error_type:
        pass
    else:
        raise AssertionError('batch should be rejected')
    assert names == original
print('Problem 11: success and validation rollback passed.')

Problem 11: success and validation rollback passed.


**Step 7 — Edge cases and engineering trade-offs.**

`list(proposed)` can itself raise for a broken iterator. Since it happens before modifying the original, the state remains unchanged. Do not treat memory allocation and arbitrary custom-list semantics as transactional; this exercise addresses the expected validation contract.

**Takeaway.** Handle only the failure you can meaningfully resolve, and let unrelated exceptions remain visible.

---

### Problem 12 — Use a sentinel to distinguish absent keys from legitimate None values

**The task.** Implement `read_setting(mapping, key, default=...)`. A missing key without a default raises a custom `MissingSetting` chained from `KeyError`. A key whose value is `None` is **present**, not missing. The function must not call `mapping.get`.

**Your goal before reading the solution:** decide which operation can fail, which exception communicates that failure, and which layer should handle it. The next cells work through the solution in small pieces.

**Step 1 — Construct the mental model.**

`None` may be a meaningful stored value. Code like `mapping.get(key) or default` conflates `None`, `0`, `False`, and the empty string with absence. A unique sentinel object preserves this distinction, and catching only `KeyError` makes errors inside a custom mapping visible.

**Step 2 — Run a minimal experiment.**

Predict what happens before executing the cell. Unlike deliberately crashing a notebook, we capture expected exceptions so **Run All** remains usable.

In [40]:
settings = {'optional': None, 'count': 0}
print('bad or-fallback:', settings.get('count') or 99)
assert (settings.get('count') or 99) == 99  # wrong for present count=0

bad or-fallback: 99


**Step 3 — Design the error contract.**

Build a sentinel for the omitted default. Try `mapping[key]` and catch `KeyError` only. If the caller supplied a default, return it; otherwise raise the meaningful domain error with `from original`.

**Step 4 — Build the reference solution incrementally.**

Read the function signature first, then trace the normal path, then locate each `raise` and `except`. Comments mark the decision boundaries.

In [41]:
_ABSENT_SETTING = object()

class MissingSetting(LookupError):
    pass

def read_setting(mapping, key, default=_ABSENT_SETTING):
    try:
        return mapping[key]
    except KeyError as original:
        if default is not _ABSENT_SETTING:
            return default
        raise MissingSetting(f'missing configuration key: {key!r}') from original

**Step 5 — Walk through why the solution works.**

An explicit `default=None` is different from an omitted default because the sentinel is unique. The chained exception retains the underlying `KeyError` for diagnostics, while the public API signals `MissingSetting`. A mapping that raises `TypeError` or another error will not be silently interpreted as missing.

**Step 6 — Exercise normal, boundary, and failure paths.**

These assertions are executable specifications, not just illustrations. If you edit the solution, rerun this cell.

In [42]:
config = {'optional': None, 'count': 0, 'enabled': False, 'label': ''}
for key in config:
    assert read_setting(config, key) == config[key]
assert read_setting(config, 'absent', None) is None
assert read_setting(config, 'absent', 0) == 0
try:
    read_setting(config, 'absent')
except MissingSetting as error:
    assert isinstance(error.__cause__, KeyError)
else:
    raise AssertionError('MissingSetting expected')
print('Problem 12: absence and false-like values distinguished.')

Problem 12: absence and false-like values distinguished.


**Step 7 — Edge cases and engineering trade-offs.**

Custom mappings may implement additional semantics (e.g., lazy lookups); catch only failures you intend to represent as missing. If keys are sensitive, avoid embedding raw keys in user-facing error messages. The example assumes key names are safe to display.

**Takeaway.** Handle only the failure you can meaningfully resolve, and let unrelated exceptions remain visible.

---

## Unit III — State changes and resource lifecycles

These problems turn exception handling into **correctness under failure**. The central question is not merely “Did we catch an error?” but “What state did we leave behind?” Each API declares what should happen before mutation, during work, and after cleanup.

### Problem 13 — Move funds atomically using a custom error taxonomy

**The task.** Model a dictionary of account balances and implement a transfer with strict validation. Unknown accounts, invalid transfer amounts, and insufficient funds should raise separate custom errors. A failed transfer may not change either balance.

**Your goal before reading the solution:** decide which operation can fail, which exception communicates that failure, and which layer should handle it. The next cells work through the solution in small pieces.

**Step 1 — Construct the mental model.**

A broad `except Exception` cannot repair a partially performed transfer. More reliable design: validate **all** inputs and constraints first, calculate the new values, then commit. Different error types let callers distinguish a missing resource, an invalid request, and insufficient balance.

**Step 2 — Run a minimal experiment.**

Predict what happens before executing the cell. Unlike deliberately crashing a notebook, we capture expected exceptions so **Run All** remains usable.

In [43]:
balances = {'A': 40, 'B': 10}
bad_copy = balances.copy()
bad_copy['A'] -= 35
print('After premature debit:', bad_copy)
assert bad_copy != balances
print('Pre-validation prevents this intermediate state from becoming public.')

After premature debit: {'A': 5, 'B': 10}
Pre-validation prevents this intermediate state from becoming public.


**Step 3 — Design the error contract.**

Define a small domain hierarchy rooted at `TransferError`, a subclass of `Exception`. Require exact positive integers (exclude `True`), two distinct known accounts, and sufficient funds. Return the updated balances without catching errors inside the operation.

**Step 4 — Build the reference solution incrementally.**

Read the function signature first, then trace the normal path, then locate each `raise` and `except`. Comments mark the decision boundaries.

In [44]:
class TransferError(Exception):
    pass
class UnknownAccount(TransferError):
    pass
class InvalidTransfer(TransferError):
    pass
class InsufficientFunds(TransferError):
    pass

def transfer(balances: dict[str, int], sender: str, receiver: str, amount: int):
    if type(amount) is not int or amount <= 0:
        raise InvalidTransfer('amount must be a positive integer')
    if sender == receiver:
        raise InvalidTransfer('sender and receiver must differ')
    for account in (sender, receiver):
        if account not in balances:
            raise UnknownAccount(account)
    if balances[sender] < amount:
        raise InsufficientFunds(f'insufficient funds in {sender!r}')
    new_sender = balances[sender] - amount
    new_receiver = balances[receiver] + amount
    # Under this example's assumptions, all validation is complete.
    balances[sender], balances[receiver] = new_sender, new_receiver
    return balances

**Step 5 — Walk through why the solution works.**

Validation failures occur before the assignment line, so both balances remain intact on any domain error. In the ordinary built-in dictionary used here, assigning two known keys with integer values is straightforward; it is **not** an actual atomic banking transaction in the presence of concurrent threads, processes, crashes, persistence, or arbitrary mapping subclasses.

**Step 6 — Exercise normal, boundary, and failure paths.**

These assertions are executable specifications, not just illustrations. If you edit the solution, rerun this cell.

In [45]:
accounts = {'A': 40, 'B': 10}
assert transfer(accounts, 'A', 'B', 15) == {'A': 25, 'B': 25}
for args, expected in [
    (('A', 'X', 2), UnknownAccount),
    (('A', 'B', 999), InsufficientFunds),
    (('A', 'B', True), InvalidTransfer),
    (('A', 'A', 1), InvalidTransfer),
]:
    snapshot = accounts.copy()
    try:
        transfer(accounts, *args)
    except expected:
        assert accounts == snapshot
    else:
        raise AssertionError(f'expected {expected.__name__}')
print('Problem 13: success and four state-preserving errors passed.')

Problem 13: success and four state-preserving errors passed.


**Step 7 — Edge cases and engineering trade-offs.**

This is a teaching model of exception safety, not a real finance implementation. Real account transfers need concurrency control, durable storage, transactions, and decimal or integer minor-unit policies. The exception design here shows how typed errors and pre-validation fit together.

**Takeaway.** Handle only the failure you can meaningfully resolve, and let unrelated exceptions remain visible.

---

### Problem 14 — Use a contextmanager that does not accidentally suppress failure

**The task.** Create `record_stage(events, name)` that records enter, success, failure, and exit. The context manager must preserve any exception raised by the body, not return a fallback value or hide it.

**Your goal before reading the solution:** decide which operation can fail, which exception communicates that failure, and which layer should handle it. The next cells work through the solution in small pieces.

**Step 1 — Construct the mental model.**

A generator-based context manager (using `@contextmanager`) resumes after `yield` when the body exits. If the body raised an exception, it is thrown at the `yield` point. Catch only to record the failure, then use bare `raise` to propagate it. Put unconditional exit recording in `finally`.

**Step 2 — Run a minimal experiment.**

Predict what happens before executing the cell. Unlike deliberately crashing a notebook, we capture expected exceptions so **Run All** remains usable.

In [46]:
@contextmanager
def probe_context(log):
    log.append('enter')
    try:
        yield
    finally:
        log.append('exit')

log = []
with probe_context(log):
    log.append('work')
assert log == ['enter', 'work', 'exit']
print(log)

['enter', 'work', 'exit']


**Step 3 — Design the error contract.**

Wrap only the `yield` in an exception handler and ensure that a bare `raise` runs after recording. Use `else` for success and `finally` for exit. Do not swallow the exception by simply reaching the end of a generator after catching it.

**Step 4 — Build the reference solution incrementally.**

Read the function signature first, then trace the normal path, then locate each `raise` and `except`. Comments mark the decision boundaries.

In [47]:
@contextmanager
def record_stage(events: list[str], name: str):
    events.append(f'enter:{name}')
    try:
        yield
    except Exception:
        events.append(f'failure:{name}')
        raise
    else:
        events.append(f'success:{name}')
    finally:
        events.append(f'exit:{name}')

**Step 5 — Walk through why the solution works.**

On normal exit, the body completes, `else` records success, and `finally` records exit. On failure, `except Exception` records the failure and uses a bare `raise`; `finally` still records exit. Catching `Exception` **only for logging and immediate re-raising** is different from pretending every exception has been fixed. Like ordinary `Exception`, this handler deliberately does not classify cancellation/control signals as recoverable.

**Step 6 — Exercise normal, boundary, and failure paths.**

These assertions are executable specifications, not just illustrations. If you edit the solution, rerun this cell.

In [48]:
events = []
with record_stage(events, 'parse'):
    events.append('body')
assert events == ['enter:parse', 'body', 'success:parse', 'exit:parse']
events = []
try:
    with record_stage(events, 'upload'):
        raise OSError('offline')
except OSError as error:
    assert str(error) == 'offline'
else:
    raise AssertionError('context manager suppressed body error')
assert events == ['enter:upload', 'failure:upload', 'exit:upload']
print('Problem 14: success and failure traces verified.')

Problem 14: success and failure traces verified.


**Step 7 — Edge cases and engineering trade-offs.**

Instrumentation must not itself fail in real systems; a broken logging sink can replace the original exception. If recording is fallible, separate best-effort diagnostics from critical cleanup. Also distinguish catching for observation from catching for recovery.

**Takeaway.** Handle only the failure you can meaningfully resolve, and let unrelated exceptions remain visible.

---

### Problem 15 — Write a class-based context manager with explicit suppression policy

**The task.** Implement a `Guard` class recording entry and exit. It suppresses only a chosen `ExpectedSkip` exception and permits any other error to propagate. Explain what the return value of `__exit__` means.

**Your goal before reading the solution:** decide which operation can fail, which exception communicates that failure, and which layer should handle it. The next cells work through the solution in small pieces.

**Step 1 — Construct the mental model.**

`__exit__(exc_type, exc, tb)` is called when leaving a `with` block. With no body exception, `exc_type` is `None`. Returning a truthy value on failure suppresses the exception; returning a false-like value leaves it propagating. Suppression should be a conscious contract, not a blanket `return True`.

**Step 2 — Run a minimal experiment.**

Predict what happens before executing the cell. Unlike deliberately crashing a notebook, we capture expected exceptions so **Run All** remains usable.

In [49]:
class ExpectedSkip(Exception):
    pass

class GuardProbe:
    def __enter__(self):
        return self
    def __exit__(self, exc_type, exc, tb):
        print('exit received:', None if exc_type is None else exc_type.__name__)
        return exc_type is ExpectedSkip

with GuardProbe():
    raise ExpectedSkip('safe to skip')
print('The selected error was suppressed intentionally.')

exit received: ExpectedSkip
The selected error was suppressed intentionally.


**Step 3 — Design the error contract.**

Log `('enter', ...)` and `('exit', exception_name)` events. Suppress exactly `ExpectedSkip` and its subclasses via `issubclass` when appropriate, not arbitrary `ValueError` or `TypeError`. Return `False` for success and for all unrelated errors.

**Step 4 — Build the reference solution incrementally.**

Read the function signature first, then trace the normal path, then locate each `raise` and `except`. Comments mark the decision boundaries.

In [50]:
class Guard:
    def __init__(self, log, label):
        self.log, self.label = log, label
    def __enter__(self):
        self.log.append(('enter', self.label))
        return self
    def __exit__(self, exc_type, exc, tb):
        label = None if exc_type is None else exc_type.__name__
        self.log.append(('exit', label))
        return exc_type is not None and issubclass(exc_type, ExpectedSkip)

**Step 5 — Walk through why the solution works.**

`__enter__` returns the resource object. `__exit__` receives exception metadata when the body fails. The boolean return is evaluated by Python to decide suppression. Because `issubclass` is guarded by `exc_type is not None`, the normal path is safe. Other exceptions stay visible.

**Step 6 — Exercise normal, boundary, and failure paths.**

These assertions are executable specifications, not just illustrations. If you edit the solution, rerun this cell.

In [51]:
log = []
with Guard(log, 'run') as guard:
    assert isinstance(guard, Guard)
assert log == [('enter', 'run'), ('exit', None)]
log = []
with Guard(log, 'run'):
    raise ExpectedSkip('expected')
assert log == [('enter', 'run'), ('exit', 'ExpectedSkip')]
log = []
try:
    with Guard(log, 'run'):
        raise TypeError('programmer error')
except TypeError:
    pass
else:
    raise AssertionError('TypeError must propagate')
assert log[-1] == ('exit', 'TypeError')
print('Problem 15: suppression policy is exact.')

Problem 15: suppression policy is exact.


**Step 7 — Edge cases and engineering trade-offs.**

A class-based context manager does not magically close resources; you must implement actual release behavior if one is owned. Never suppress an unknown error just to keep execution moving. `issubclass` handles subclasses of the explicitly selected exception without catching unrelated classes.

**Takeaway.** Handle only the failure you can meaningfully resolve, and let unrelated exceptions remain visible.

---

### Problem 16 — Attach context to a raised exception without changing its type

**The task.** Build a validation pipeline where the deepest function raises `ValueError`. The outer layer adds record context using `add_note()` and re-raises the **same exception object**. Do not leak a password field in the note.

**Your goal before reading the solution:** decide which operation can fail, which exception communicates that failure, and which layer should handle it. The next cells work through the solution in small pieces.

**Step 1 — Construct the mental model.**

Python 3.11 adds `BaseException.add_note(str)`. Notes enrich diagnostic output while preserving the type, identity, and traceback of the original exception. They live in `__notes__`, which can be inspected programmatically. Include safe identifiers only; do not attach secrets to exceptions.

**Step 2 — Run a minimal experiment.**

Predict what happens before executing the cell. Unlike deliberately crashing a notebook, we capture expected exceptions so **Run All** remains usable.

In [52]:
error = ValueError('invalid age')
error.add_note('record_id=42')
print('type:', type(error).__name__)
print('notes:', error.__notes__)
assert error.__notes__ == ['record_id=42']

type: ValueError
notes: ['record_id=42']


**Step 3 — Design the error contract.**

Write a strict age validator and a record-level wrapper. The wrapper catches `ValueError` only, attaches the public record ID, and uses bare `raise`. Keep the dictionary's secret field out of the note and message.

**Step 4 — Build the reference solution incrementally.**

Read the function signature first, then trace the normal path, then locate each `raise` and `except`. Comments mark the decision boundaries.

In [53]:
def require_adult_age(age):
    if type(age) is not int or age < 18:
        raise ValueError('age must be an integer of at least 18')
    return age

def validate_public_record(record):
    try:
        return require_adult_age(record['age'])
    except ValueError as error:
        error.add_note(f"record_id={record['id']!r}")
        raise

**Step 5 — Walk through why the solution works.**

The wrapper does not convert or replace the `ValueError`. It adds one contextual note, then re-raises. `KeyError` for a missing `age` or `id` is not caught by this particular handler and therefore is not mislabeled as an age-validation error. Notes are for diagnostic context, not a substitute for dedicated exception attributes when consumers need a structured API.

**Step 6 — Exercise normal, boundary, and failure paths.**

These assertions are executable specifications, not just illustrations. If you edit the solution, rerun this cell.

In [54]:
record = {'id': 'public-7', 'age': 15, 'password': 'DO_NOT_LOG_ME'}
try:
    validate_public_record(record)
except ValueError as error:
    assert error.__notes__ == ["record_id='public-7'"]
    assert 'DO_NOT_LOG_ME' not in str(error)
    assert all('DO_NOT_LOG_ME' not in note for note in error.__notes__)
    print('Notes:', error.__notes__)
else:
    raise AssertionError('invalid age must fail')
assert validate_public_record({'id': 'ok', 'age': 20}) == 20
print('Problem 16: error identity strategy and redaction passed.')

Notes: ["record_id='public-7'"]
Problem 16: error identity strategy and redaction passed.


**Step 7 — Edge cases and engineering trade-offs.**

Notes can appear when tracebacks are printed, so avoid personal, secret, or regulated information. Reusing the *same exception instance* across multiple attempts may accumulate notes; prefer a fresh error per operation. This is Python 3.11+ functionality.

**Takeaway.** Handle only the failure you can meaningfully resolve, and let unrelated exceptions remain visible.

---

## Unit IV — Advanced exception mechanisms and testing

The original lecture focuses on single raised exceptions and ordinary handlers. In Python 3.11+, exception groups support multiple concurrent failures; the following problems introduce their syntax carefully. We also practice testing exception contracts without making the notebook crash.

### Problem 17 — Partition a nested ExceptionGroup without losing its structure

**The task.** Take a nested group containing `ValueError`, `OSError`, and `TypeError`. Split out the `ValueError` leaves while retaining the nested grouping. Verify the original group remains unchanged and the remainder contains only non-ValueError leaves.

**Your goal before reading the solution:** decide which operation can fail, which exception communicates that failure, and which layer should handle it. The next cells work through the solution in small pieces.

**Step 1 — Construct the mental model.**

An `ExceptionGroup` stores multiple ordinary exceptions, potentially nested. `.split(ValueError)` returns a pair `(matches, rest)` with the grouping structure retained as appropriate. Neither group is mutated; one of the returned values may be `None` when no leaves match.

**Step 2 — Run a minimal experiment.**

Predict what happens before executing the cell. Unlike deliberately crashing a notebook, we capture expected exceptions so **Run All** remains usable.

In [55]:
group_probe = ExceptionGroup('root', [
    ValueError('a'),
    ExceptionGroup('nested', [OSError('disk'), ValueError('b')]),
])
matching, remaining = group_probe.split(ValueError)
print('matching root:', matching.message)
print('remaining root:', remaining.message)
print('matching top-level entries:', len(matching.exceptions))

matching root: root
remaining root: root
matching top-level entries: 2


**Step 3 — Design the error contract.**

Write a recursive leaf collector for inspection only. Use built-in `split` to perform the actual partition and validate leaves by type. Never flatten the original group in the production split, because nested context can carry meaning.

**Step 4 — Build the reference solution incrementally.**

Read the function signature first, then trace the normal path, then locate each `raise` and `except`. Comments mark the decision boundaries.

In [56]:
def exception_leaves(error):
    if isinstance(error, BaseExceptionGroup):
        leaves = []
        for child in error.exceptions:
            leaves.extend(exception_leaves(child))
        return leaves
    return [error]

def partition_validation_errors(group: ExceptionGroup):
    return group.split(ValueError)

**Step 5 — Walk through why the solution works.**

The collector is a test aid: it walks the nested structure to inspect leaf types. `.split()` preserves subgroup structure where possible and returns two group objects or `None`. Unlike manually concatenating errors, this preserves the original relationships and makes category-specific downstream handling possible.

**Step 6 — Exercise normal, boundary, and failure paths.**

These assertions are executable specifications, not just illustrations. If you edit the solution, rerun this cell.

In [57]:
original = ExceptionGroup('batch', [
    ValueError('bad A'),
    ExceptionGroup('secondary', [OSError('offline'), ValueError('bad B')]),
    TypeError('bug'),
])
validations, other = partition_validation_errors(original)
assert [type(e) for e in exception_leaves(validations)] == [ValueError, ValueError]
assert [type(e) for e in exception_leaves(other)] == [OSError, TypeError]
assert len(exception_leaves(original)) == 4
none_match, all_rest = ExceptionGroup('only IO', [OSError('x')]).split(ValueError)
assert none_match is None and all_rest is not None
print('Problem 17: nested split and no-match case passed.')

Problem 17: nested split and no-match case passed.


**Step 7 — Edge cases and engineering trade-offs.**

`BaseExceptionGroup` may also contain direct `BaseException` subclasses. This exercise constructs `ExceptionGroup`, whose leaves derive from `Exception`. Do not assume group nesting maps to chronological order, and do not discard the remainder after handling one category.

**Takeaway.** Handle only the failure you can meaningfully resolve, and let unrelated exceptions remain visible.

---

### Problem 18 — Handle two categories from a group using except*

**The task.** Write a function that raises a group containing two `ValueError`s and one `OSError`. Catch the value errors and the OS error in separate `except*` branches, recording all three. Explain why ordinary `except ValueError` is not enough.

**Your goal before reading the solution:** decide which operation can fail, which exception communicates that failure, and which layer should handle it. The next cells work through the solution in small pieces.

**Step 1 — Construct the mental model.**

`except*` matches individual leaves within exception groups. It can run multiple branches for one raised group. A standard `except ValueError` handler does not unpack an `ExceptionGroup` to catch its contained values. Python requires separate `try` statements if you want to mix plain `except` and `except*` styles; do not combine them on one `try`.

**Step 2 — Run a minimal experiment.**

Predict what happens before executing the cell. Unlike deliberately crashing a notebook, we capture expected exceptions so **Run All** remains usable.

In [58]:
example_group = ExceptionGroup('mixed', [ValueError('x'), OSError('y')])
try:
    raise example_group
except ExceptionGroup as whole:
    print('ordinary except caught the group, not a ValueError leaf:',
          type(whole).__name__)

ordinary except caught the group, not a ValueError leaf: ExceptionGroup


**Step 3 — Design the error contract.**

Create the group in a helper, then use a `try` with two `except*` clauses. Each clause receives an `ExceptionGroup` subgroup (not an individual leaf). Collect messages from each subgroup and sort them to make testing independent of display formatting.

**Step 4 — Build the reference solution incrementally.**

Read the function signature first, then trace the normal path, then locate each `raise` and `except`. Comments mark the decision boundaries.

In [59]:
def handle_mixed_group():
    captured = {'validation': [], 'io': []}
    try:
        raise ExceptionGroup('failures', [
            ValueError('v1'), OSError('disk'), ValueError('v2')
        ])
    except* ValueError as subgroup:
        captured['validation'].extend(str(e) for e in subgroup.exceptions)
    except* OSError as subgroup:
        captured['io'].extend(str(e) for e in subgroup.exceptions)
    return captured

**Step 5 — Walk through why the solution works.**

Both handlers run on their respective leaves. The `except* ValueError` branch sees a subgroup containing two errors; the `except* OSError` branch sees one. Any unmatched leaf would remain unhandled and propagate as a group, so this implementation purposely covers every leaf its helper raises.

**Step 6 — Exercise normal, boundary, and failure paths.**

These assertions are executable specifications, not just illustrations. If you edit the solution, rerun this cell.

In [60]:
captured = handle_mixed_group()
assert sorted(captured['validation']) == ['v1', 'v2']
assert captured['io'] == ['disk']
print('Problem 18:', captured)

Problem 18: {'validation': ['v1', 'v2'], 'io': ['disk']}


**Step 7 — Edge cases and engineering trade-offs.**

You cannot use `return`, `break`, or `continue` inside an `except*` block. The implementation returns after the whole `try` statement. In general, do not use `except* Exception` just to hide an unknown remainder.

**Takeaway.** Handle only the failure you can meaningfully resolve, and let unrelated exceptions remain visible.

---

### Problem 19 — Test exception contracts using assertRaises and diagnostic assertions

**The task.** Build a strict index-selector and an executable test matrix with `unittest`. Verify both the exception class and informative text; include ordinary success tests and boundaries.

**Your goal before reading the solution:** decide which operation can fail, which exception communicates that failure, and which layer should handle it. The next cells work through the solution in small pieces.

**Step 1 — Construct the mental model.**

Good error-handling tests check: normal output, relevant boundaries, exception type, error details, and whether unrelated errors propagate. `unittest.TestCase.assertRaisesRegex` captures an exception and checks the message. Avoid asserting an entire rendered traceback or hard-coding Python’s version-dependent built-in error strings.

**Step 2 — Run a minimal experiment.**

Predict what happens before executing the cell. Unlike deliberately crashing a notebook, we capture expected exceptions so **Run All** remains usable.

In [61]:
import unittest

with unittest.TestCase().assertRaises(ValueError):
    int('not-a-number')
print('A context-managed assertion checks the expected exception.')

A context-managed assertion checks the expected exception.


**Step 3 — Design the error contract.**

The API accepts a nonnegative integer index, but excludes bool. It rejects invalid types with `TypeError` and out-of-range values with `IndexError`, including negative indexes even though Python lists normally accept them. Build table-driven tests that enforce this intentional contract.

**Step 4 — Build the reference solution incrementally.**

Read the function signature first, then trace the normal path, then locate each `raise` and `except`. Comments mark the decision boundaries.

In [62]:
def strict_select(values, index):
    if type(index) is not int:
        raise TypeError('index must be an integer (bool excluded)')
    if index < 0 or index >= len(values):
        raise IndexError(f'index {index} outside valid range')
    return values[index]

class StrictSelectTests(unittest.TestCase):
    def test_success(self):
        self.assertEqual(strict_select(['a', 'b'], 1), 'b')
        self.assertEqual(strict_select(['a'], 0), 'a')

    def test_type_errors(self):
        for bad in (True, 1.0, '1', None):
            with self.subTest(value=repr(bad)):
                with self.assertRaisesRegex(TypeError, 'index must be'):
                    strict_select(['a'], bad)

    def test_index_errors(self):
        for bad in (-1, 2):
            with self.subTest(index=bad):
                with self.assertRaisesRegex(IndexError, 'outside valid range'):
                    strict_select(['a', 'b'], bad)

**Step 5 — Walk through why the solution works.**

`subTest` identifies the failing input in a table-driven test. A `TestCase` checks exactly what the API promises: error **type** and a stable substring of its own message, not implementation-specific traceback formatting. Normal tests ensure the function has not become a validator that only raises.

**Step 6 — Exercise normal, boundary, and failure paths.**

These assertions are executable specifications, not just illustrations. If you edit the solution, rerun this cell.

In [63]:
suite = unittest.defaultTestLoader.loadTestsFromTestCase(StrictSelectTests)
result = unittest.TextTestRunner(verbosity=0).run(suite)
assert result.wasSuccessful() and result.testsRun == 3
print('Problem 19: 3 test methods passed, including subtests.')

----------------------------------------------------------------------
Ran 3 tests in 0.001s

OK


Problem 19: 3 test methods passed, including subtests.


**Step 7 — Edge cases and engineering trade-offs.**

Avoid `assertRaises(Exception)` when your API promises a specific exception: it can accept the wrong failure. Testing only the “happy path” similarly misses the reason this API exists. Keep test code dependency-free so the notebook runs in a clean Python kernel.

**Takeaway.** Handle only the failure you can meaningfully resolve, and let unrelated exceptions remain visible.

---

### Problem 20 — Separate error conversion from error presentation

**The task.** Implement a service boundary that accepts an integer ID string and returns a record. Reject malformed IDs as `InvalidRecordID` (chained from `ValueError`), unknown IDs as `RecordNotFound`, and let unexpected storage defects propagate. Then format only the two expected domain errors for a user.

**Your goal before reading the solution:** decide which operation can fail, which exception communicates that failure, and which layer should handle it. The next cells work through the solution in small pieces.

**Step 1 — Construct the mental model.**

A useful architecture has three layers: input conversion, domain lookup, and user-facing presentation. Each layer owns only the failures it understands. A storage `TypeError` is not a “record not found” event. The UI catches a small tuple of documented domain errors, never `Exception` just to produce reassuring text.

**Step 2 — Run a minimal experiment.**

Predict what happens before executing the cell. Unlike deliberately crashing a notebook, we capture expected exceptions so **Run All** remains usable.

In [64]:
sample = {'1': 'Alice'}
try:
    int('not-an-id')
except ValueError:
    print('Conversion failure and missing key are different problems.')
try:
    sample['999']
except KeyError:
    print('Lookup failure occurs after valid conversion.')

Conversion failure and missing key are different problems.
Lookup failure occurs after valid conversion.


**Step 3 — Design the error contract.**

Create two domain error classes. Translate conversion errors with an explicit cause; test for missing IDs after conversion; make the presenter handle only those two classes. A simulated broken store can prove that unexpected defects escape.

**Step 4 — Build the reference solution incrementally.**

Read the function signature first, then trace the normal path, then locate each `raise` and `except`. Comments mark the decision boundaries.

In [65]:
class InvalidRecordID(ValueError):
    pass
class RecordNotFound(LookupError):
    pass

def retrieve_record(id_text, store):
    try:
        record_id = int(id_text)
    except ValueError as original:
        raise InvalidRecordID('record ID must be an integer') from original
    try:
        return store[record_id]
    except KeyError as original:
        raise RecordNotFound(f'unknown record ID: {record_id}') from original

def present_record(id_text, store):
    try:
        return f'Record: {retrieve_record(id_text, store)}'
    except (InvalidRecordID, RecordNotFound) as error:
        return f'Unable to display record: {error}'

**Step 5 — Walk through why the solution works.**

A malformed integer string is translated to `InvalidRecordID`; a valid but missing ID translates to `RecordNotFound`. Each translation carries its own cause. The presentation layer catches the documented errors and returns a message, while leaving an unexpected storage `TypeError` visible to a developer.

**Step 6 — Exercise normal, boundary, and failure paths.**

These assertions are executable specifications, not just illustrations. If you edit the solution, rerun this cell.

In [66]:
data = {1: 'Alice', 2: 'Bob'}
assert present_record('1', data) == 'Record: Alice'
assert 'integer' in present_record('oops', data)
assert 'unknown' in present_record('999', data)
try:
    retrieve_record('oops', data)
except InvalidRecordID as error:
    assert isinstance(error.__cause__, ValueError)
try:
    retrieve_record('99', data)
except RecordNotFound as error:
    assert isinstance(error.__cause__, KeyError)
class BrokenStore:
    def __getitem__(self, key):
        raise TypeError('storage implementation broke')
try:
    present_record('1', BrokenStore())
except TypeError as error:
    assert 'implementation' in str(error)
else:
    raise AssertionError('storage defect must propagate')
print('Problem 20: three-layer contract passed.')

Problem 20: three-layer contract passed.


**Step 7 — Edge cases and engineering trade-offs.**

`int(True)` succeeds, so a string-only input contract would also require explicit type validation; here the contract intentionally permits any object accepted by `int()`. In a public API, document permitted types and sanitize user messages separately from developer diagnostics.

**Takeaway.** Handle only the failure you can meaningfully resolve, and let unrelated exceptions remain visible.

---

## Unit V — Extended capstone: a dependency resolver built through small steps

The capstone combines the source lesson’s concepts: specific `KeyError` vs domain errors, propagation across recursive calls, custom exception classes, narrow `try`, and preserving diagnostic causes. We do **not** catch everything. Every part is taught separately before the final integration.

### Problem 21 — CAPSTONE: Resolve a dependency graph with useful errors

**Scenario.** A tiny task runner stores dependencies in a dictionary:

```python
{
    'compile': ('parse', 'typecheck'),
    'typecheck': ('parse',),
    'parse': (),
}
```

Write `resolve_order(graph, targets)` so each dependency appears **before** the task that requires it, and each task is emitted once. A missing referenced task raises `UnknownTask` chained from `KeyError`. A dependency cycle raises `DependencyCycle` with its full cycle path. A wrong dependency type raises `InvalidDependencyGraph`. The original graph must never be mutated.

**Acceptance cases:** shared dependencies, multiple targets, missing targets, missing child dependencies, self-cycles, indirect cycles, invalid dependency containers, and no-op empty targets.

**Capstone Step 1 — Trace a correct graph on paper.**

For `compile → {parse, typecheck}`, DFS visits `parse`, marks it completed, then visits `typecheck` (which sees `parse` is already complete), and only then outputs `compile`. The resulting order is `parse, typecheck, compile`. We need two concepts:

- `active`: tasks on the **current recursive path**. Encountering an active task means a cycle.
- `done`: tasks whose dependencies and task have already been emitted. Encountering one means safe reuse.

A single `visited` set cannot distinguish those two states, so it can miss cycles.

In [67]:
example_graph = {'compile': ('parse', 'typecheck'),
                 'typecheck': ('parse',), 'parse': ()}
print('Tasks:', sorted(example_graph))
print('Expected valid order:', ['parse', 'typecheck', 'compile'])

Tasks: ['compile', 'parse', 'typecheck']
Expected valid order: ['parse', 'typecheck', 'compile']


**Capstone Step 2 — Specify three distinct error types.**

We avoid catching `Exception` at the top. Each error has a different resolution:

- `UnknownTask`: supply a missing task definition or correct a typo.
- `DependencyCycle`: change the dependency graph so it becomes acyclic.
- `InvalidDependencyGraph`: use a tuple/list of task names instead of an invalid dependency container.

The public exception classes are stable API signals. `UnknownTask` stores the task name, `DependencyCycle` stores the cycle path, and the underlying `KeyError` is preserved via explicit chaining.

In [68]:
class DependencyError(Exception):
    """Base class for expected dependency-configuration failures."""

class UnknownTask(DependencyError):
    def __init__(self, name):
        self.name = name
        super().__init__(f'unknown task: {name!r}')

class DependencyCycle(DependencyError):
    def __init__(self, path):
        self.path = tuple(path)
        super().__init__('dependency cycle: ' + ' -> '.join(map(str, path)))

class InvalidDependencyGraph(DependencyError):
    pass

**Capstone Step 3 — Choose the error boundary.**

There is exactly one dictionary lookup that translates `KeyError → UnknownTask`; place its handler *around that lookup alone*. Recursion and validation should remain outside the handler. Otherwise an unrelated `KeyError` from a later operation might masquerade as an unknown task. Reject a string as the entire dependency container (`'parse'` is not a tuple of dependency names), and reject non-string names as invalid configuration rather than treating them as missing keys.

In [69]:
def read_dependencies(graph, name):
    if not isinstance(name, str):
        raise InvalidDependencyGraph(f'task name must be str: {name!r}')
    try:
        dependencies = graph[name]
    except KeyError as original:
        raise UnknownTask(name) from original
    if not isinstance(dependencies, (list, tuple)):
        raise InvalidDependencyGraph(
            f'dependencies for {name!r} must be list or tuple')
    if any(not isinstance(item, str) for item in dependencies):
        raise InvalidDependencyGraph(f'dependencies for {name!r} must be strings')
    return dependencies

assert read_dependencies(example_graph, 'parse') == ()
try:
    read_dependencies(example_graph, 'missing')
except UnknownTask as error:
    assert error.name == 'missing'
    assert isinstance(error.__cause__, KeyError)
print('Task reading and KeyError translation work in isolation.')

Task reading and KeyError translation work in isolation.


**Capstone Step 4 — Implement DFS without an all-purpose exception handler.**

A task is marked active when entered and removed when its full subtree is processed. The `active` list retains visitation order so it can construct the precise cycle segment. The `done` set prevents repeated output and repeated walks through shared dependencies. This solution is iterative only over each task's dependencies; Python recursion depth still applies to extremely deep graphs.

In [70]:
def resolve_order(graph, targets):
    order = []
    done = set()
    active = []

    def visit(name):
        if name in active:
            start = active.index(name)
            raise DependencyCycle(active[start:] + [name])
        if name in done:
            return
        dependencies = read_dependencies(graph, name)
        active.append(name)
        try:
            for dependency in dependencies:
                visit(dependency)
        finally:
            # Restore internal traversal state even when child visiting fails.
            active.pop()
        done.add(name)
        order.append(name)

    for target in targets:
        visit(target)
    return order

**Capstone Step 5 — Explain the `finally` in the recursive implementation.**

`active.append(name)` establishes an invariant: the list represents the current call path. The `finally` clause removes `name` whether children succeed or throw. It does not `return` or suppress errors; therefore the first meaningful `UnknownTask`, `DependencyCycle`, or `InvalidDependencyGraph` reaches the caller. On a successful visit, `done.add` and `order.append` occur after traversal, which puts dependencies first. The graph is only read, never mutated.

**Capstone Step 6 — Execute a behavior matrix.**

Tests check more than error type: cycles must expose a usable path, missing tasks must retain their original `KeyError`, and a failed traversal may not mutate the input graph. Each failure is caught deliberately inside a test so the notebook keeps running.

In [71]:
from copy import deepcopy

graph = {'compile': ('parse', 'typecheck'),
         'typecheck': ('parse',), 'parse': (), 'package': ('compile',)}
original_graph = deepcopy(graph)
assert resolve_order(graph, ['compile']) == ['parse', 'typecheck', 'compile']
assert resolve_order(graph, ['compile', 'package']) == [
    'parse', 'typecheck', 'compile', 'package']
assert resolve_order(graph, []) == []
assert graph == original_graph

error_cases = [
    ({'a': ('missing',)}, ['a'], UnknownTask),
    ({'a': ('a',)}, ['a'], DependencyCycle),
    ({'a': ('b',), 'b': ('c',), 'c': ('a',)}, ['a'], DependencyCycle),
    ({'a': 'b'}, ['a'], InvalidDependencyGraph),
    ({'a': (123,)}, ['a'], InvalidDependencyGraph),
]
for bad_graph, targets, expected_type in error_cases:
    snapshot = deepcopy(bad_graph)
    try:
        resolve_order(bad_graph, targets)
    except expected_type as error:
        assert bad_graph == snapshot
        if isinstance(error, UnknownTask):
            assert isinstance(error.__cause__, KeyError)
        if isinstance(error, DependencyCycle):
            assert error.path[0] == error.path[-1]
    else:
        raise AssertionError(f'expected {expected_type.__name__}')
print('Capstone: order, shared deps, missing tasks, two cycles, invalid graphs passed.')

Capstone: order, shared deps, missing tasks, two cycles, invalid graphs passed.


**Capstone Step 7 — Apply fault-oriented reasoning.**

Try these modifications mentally before you edit code:

1. Move the `try/except KeyError` to cover the entire recursive `visit` function. What unrelated failures might it misclassify?
2. Replace `active` and `done` with one set. What happens for `a → b → a`?
3. Move `done.add(name)` before visiting children. Will cycles be visible?
4. Add `return 'failed'` inside `finally`. Which pending exception becomes hidden?

The implementation keeps the task lookup's `try` narrow, separates active from completed tasks, and uses `finally` solely to restore its internal invariant.

In [72]:
# An additional, separate test demonstrates that independent targets
# share dependencies without duplicate entries.
shared = {'left': ('base',), 'right': ('base',), 'base': ()}
assert resolve_order(shared, ['left', 'right']) == ['base', 'left', 'right']

# A missing target fails just like a missing child dependency.
try:
    resolve_order(shared, ['unknown_target'])
except UnknownTask as error:
    assert error.name == 'unknown_target'
    assert isinstance(error.__cause__, KeyError)
else:
    raise AssertionError('Unknown target must fail')
print('Capstone stretch cases passed.')

Capstone stretch cases passed.


**Capstone Step 8 — Production considerations.**

This resolver is a deterministic teaching example. Real task runners often need iterative traversal for very deep graphs, concurrency protection if the graph can change while being read, target deduplication policy, an error-reporting protocol for multiple independent roots, and separate execution semantics. Those concerns should be added deliberately rather than being obscured by `except Exception: pass`.

---

## Final review — Write your own exception-handling checklist

Before finalizing a function, answer these questions in order:

1. What exact failure is expected, and what type represents it?
2. Which **smallest operation** should sit inside the `try` suite?
3. Is there a meaningful recovery, or should the exception propagate?
4. If translating, did I use `raise NewError(...) from original` and preserve useful context?
5. What state is visible if the operation fails halfway through?
6. Does `finally` clean up without hiding a pending failure?
7. Do tests cover success, boundaries, expected errors, and unexpected defects?
8. Can messages or notes leak a secret?

**Big idea from your source lesson, now applied at larger scale:** Exception hierarchies make handling flexible, but catching too widely can disguise unrelated errors. Prefer narrow, intentional contracts; let the call stack carry failures to the layer that can actually handle them.

In [73]:
# End-to-end notebook checkpoint (a few key cross-problem contracts).
assert validate_age(30) == 30
assert classify_lookup(lambda: 'ok') == 'success'
assert parse_integer_fields('2;3') == [2, 3]
assert list(exact_pairs(range(4))) == [(0, 1), (2, 3)]
assert read_setting({'x': None}, 'x') is None
assert resolve_order({'root': ('leaf',), 'leaf': ()}, ['root']) == ['leaf', 'root']
print('FINAL CHECKPOINT PASSED: tutorial definitions remain usable.')

FINAL CHECKPOINT PASSED: tutorial definitions remain usable.
